In [12]:
# Build main diff-in-diff analysis data

import os
import pandas as pd
import numpy as np
import dotenv

import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

dotenv.load_dotenv(dotenv.find_dotenv())

ROOT_PATH = os.getenv("ROOT_PATH")
MY_DATA_PATH = os.getenv("MY_DATA_PATH")
RAW_DATA_PATH = os.getenv("RAW_DATA_PATH")

OUTPUT_FILEPATH = os.path.join(MY_DATA_PATH, "processed_data/tax_analysis_panel.parquet")

MIN_YEAR = 2010
MAX_YEAR = 2023



In [13]:
# Load data

regs_df_1 = pd.read_csv(os.path.join(RAW_DATA_PATH, "sales-analysis-redfin/data/best_treatment_dates_2026-07.csv"))
regs_df_2 = pd.read_csv(os.path.join(RAW_DATA_PATH, "sales-analysis-redfin/data/str_dates_cities_51_100.csv"))
fisc_df = pd.read_excel(os.path.join(RAW_DATA_PATH, "lincoln-institute/FiSC-Full-Dataset-2023-Update.xlsx"), sheet_name="Data")
zhvi_df = pd.read_csv(os.path.join(RAW_DATA_PATH, "zhvi/City_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"))

zhvi_xwalk = pd.read_csv(os.path.join(MY_DATA_PATH, "raw_data/regs_zhvi_xwalk.csv"))
fisc_xwalk = pd.read_csv(os.path.join(MY_DATA_PATH, "raw_data/regs_fisc_xwalk.csv"))


In [14]:
# Cleaning regs_df_1

regs_df_1_clean = regs_df_1[['city', 'state', 'best_passage', 'best_enforcement']].rename(
    columns={
        'best_passage': 'passage_date',
        'best_enforcement': 'effective_date'
    }
)
regs_df_1_clean['passage_date'] = pd.to_datetime(regs_df_1_clean['passage_date'])
regs_df_1_clean['effective_date'] = pd.to_datetime(regs_df_1_clean['effective_date'])


In [15]:
# Cleaning regs_df_2

regs_df_2_clean = regs_df_2[['city', 'state', 'passage_date', 'effective_date']]
regs_df_2_clean['passage_date'] = pd.to_datetime(regs_df_2_clean['passage_date'])
regs_df_2_clean['effective_date'] = pd.to_datetime(regs_df_2_clean['effective_date'])


In [16]:
# Cleaning fisc_df

fisc_df_clean = fisc_df.rename(columns={
    'id_city': 'fisc_id'
})

In [17]:
# Reshape/clean zhvi data to long by year

id_cols = ['RegionID', 'SizeRank', 'RegionName', 'RegionType',
           'StateName', 'State', 'Metro', 'CountyName']

zhvi_long = zhvi_df.melt(
    id_vars=id_cols,
    var_name = 'date',
    value_name = 'ZHVI'
)

zhvi_long['date'] = pd.to_datetime(zhvi_long['date'])
zhvi_long['year'] = zhvi_long['date'].dt.year

zhvi_long = zhvi_long.groupby(id_cols + ['year']).agg({'ZHVI': 'mean'}).reset_index()

zhvi_long = zhvi_long.rename(columns={
    'RegionID': 'zhvi_id'
})

zhvi_long = zhvi_long[['zhvi_id', 'year', 'ZHVI']]


In [18]:
# Merge the data

df = pd.concat([regs_df_1_clean, regs_df_2_clean], ignore_index=True)

df['id'] = df.index  # unique identifier for each city

df = df.merge(fisc_xwalk[['city', 'state', 'fisc_id']], on=['city', 'state'], how='inner').reset_index(drop=True)

df = df.merge(zhvi_xwalk[['city', 'state', 'zhvi_id']], on=['city', 'state'], how='inner').reset_index(drop=True)

df = df.merge(fisc_df_clean, on=['fisc_id'], how='inner').reset_index(drop=True)

df = df.merge(zhvi_long, on=['zhvi_id', 'year'], how='left').reset_index(drop=True)


In [19]:
# Check panel structure

assert df[['city', 'state', 'year']].duplicated().sum() == 0
n_city = len(df.groupby(['city', 'state']))
n_year = len(df['year'].unique())
assert len(df) == n_city * n_year

In [20]:
# Time variables

df['passage_year'] = df['passage_date'].dt.year
df['effective_year'] = df['effective_date'].dt.year

df['years_after_passage'] = df['year'] - df['passage_year']
df['years_after_effective'] = df['year'] - df['effective_year']

# change effective/passage year to 0 for cities without effective/passage years
# (standard convention for CSDID package in R)

df.loc[df['effective_year'].isna(), 'effective_year'] = 0
df.loc[df['passage_year'].isna(), 'passage_year'] = 0

df.loc[df['effective_year']==0, 'years_after_effective'] = 0
df.loc[df['passage_year']==0, 'years_after_passage'] = 0


In [21]:
# Year selection

df = df[df['year']>=MIN_YEAR]
df = df[df['year']<=MAX_YEAR]
df = df.reset_index(drop=True)

In [22]:
# output dataframe for analysis

df.to_parquet(OUTPUT_FILEPATH)